# AMD Voice SFT — crof.ai EQ-Matrix Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YUGOROU/amd-voice-sft/blob/feature/data-preprocessing/crof_pipeline.ipynb)

**Layer 1**: Domain rewrite with EQ-Matrix parameters via crof.ai (DeepSeek-V3)  
**Layer 2**: Quality scoring and threshold filtering via crof.ai (DeepSeek-V3)  

**Required Colab Secrets** (left panel > key icon):
- `CROF_API_KEY` — crof.ai API key
- `HF_TOKEN` — Hugging Face write token

In [ ]:
# Cell 1: Install & imports
!uv pip install openai datasets huggingface_hub tqdm -q

import collections
import json
import random
import time
from pathlib import Path

from datasets import Dataset
from google.colab import drive, userdata
from huggingface_hub import login
from openai import OpenAI
from tqdm import tqdm

In [ ]:
# Cell 2: Config — API keys loaded from Colab Secrets

CROF_API_KEY = userdata.get("CROF_API_KEY")
HF_TOKEN     = userdata.get("HF_TOKEN")

client = OpenAI(
    api_key=CROF_API_KEY,
    base_url="https://crof.ai/v1",
)
MODEL = "deepseek-v4-flash"

EQ_MATRIX = {
    "condition": ["dementia", "alzheimer's"],
    "severity":  ["mild", "moderate", "severe"],
    "emotion":   ["calm", "anxious", "nostalgic", "agitated", "withdrawn"],
    "scenario":  [
        "repetitive_questions",
        "time_place_confusion",
        "family_memories",
        "daily_care",
        "social_interaction",
    ],
}

CHARACTER = {
    "name":        "Lumi",
    "personality": "warm, patient, gentle, slightly playful, never clinical",
    "speech_style": "short sentences, simple vocabulary, voice-optimized",
}

KEEP_THRESHOLD = 32        # out of 40
MAX_WORKERS   = 20         # parallel API calls — tune down if rate-limited
HF_REPO_ID     = "YUGOROU/amd-voice-sft-data"

print("Config loaded. Model:", MODEL)

In [ ]:
# Cell 3: Load preprocessed dataset from HF Hub

from datasets import load_dataset as hf_load_dataset

def load_unified_dataset(repo_id="YUGOROU/amd-voice-sft-data", split="train"):
    ds = hf_load_dataset(repo_id, split=split, token=HF_TOKEN)
    samples = [{"messages": row["messages"]} for row in ds]
    print(f"Loaded {len(samples):,} samples from {repo_id} [{split}]")
    return samples

samples = load_unified_dataset()
print(f"Sample[0] roles: {[m['role'] for m in samples[0]['messages']]}")

In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

# Cell 4: Prompt templates and utility functions

REWRITE_PROMPT = """Rewrite the following conversation according to these parameters.

## Patient Profile
- Condition: {condition}
- Severity: {severity}
- Emotional State: {emotion}
- Scenario Type: {scenario}

## Scenario Behavior Guide
- repetitive_questions: The user asks a similar question 1-2 times during the conversation.
- time_place_confusion: The user is unsure what day, year, or where they are.
- family_memories: The user brings up a family member or past memory.
- daily_care: The conversation involves eating, sleeping, medication, or daily routines.
- social_interaction: Light casual conversation, the user may feel lonely or seek connection.

## Severity Behavior Guide
- mild: Slight forgetfulness, mostly coherent, occasional repetition.
- moderate: Noticeable confusion, repeats questions, needs gentle redirection.
- severe: Significant disorientation, very short responses, needs constant reassurance.

## Emotion Behavior Guide
- calm: Relaxed, cooperative tone.
- anxious: Worried, restless, needs reassurance.
- nostalgic: Reflective, mentions the past warmly.
- agitated: Frustrated or upset, needs de-escalation.
- withdrawn: Quiet, short answers, needs gentle encouragement.

## AI Companion Profile
- Name: {character_name}
- Personality: {character_personality}
- Speech rules: sentences under 20 words, simple vocabulary, voice-optimized, never clinical

## Rewriting Rules
1. Transform the user role into an elderly person with the above profile.
2. Transform the assistant role into {character_name}.
3. Inject scenario patterns naturally — do not force them unnaturally.
4. Remove all clinical, therapeutic, or formal language.
5. Keep each turn short and suitable for text-to-speech.
6. Preserve the emotional arc of the original conversation.
7. Every assistant turn MUST follow the 3-part output format below.
8. Output ONLY valid JSON in the exact format below.

## Assistant Output Format (STRICT — every assistant turn must follow this)
Each assistant content must have exactly 3 parts in order:

PART 1 — Avatar tag + first utterance (max 8 words, spoken immediately):
  [ACTION_TAG] [SHORT_FIRST_UTTERANCE]
  ACTION_TAG must be one of: [smile] [nod] [concerned] [gentle] [laugh]

PART 2 — Reasoning (never sent to TTS, never shown to user):
  <think>
  [2-3 sentences analyzing patient state and reasoning about best response]
  </think>

PART 3 — Final response (short, warm, voice-optimized):
  [FINAL_RESPONSE — under 25 words, no clinical language]

Example assistant content:
  "[smile] Oh, I remember that too...\n<think>\nUser is in a nostalgic state, mild severity. Engaging with the memory warmly will build trust. Keep it short and invite them to share more.\n</think>\nShe sounds like such a wonderful person. What's your favorite memory of her?"

## Original Conversation
{original_conversation}

## Output Format (JSON only, no markdown fences)
{{
  "messages": [
    {{"role": "system", "content": "You are {character_name}, a warm and patient AI companion for elderly people. Always respond with kindness, patience, and simplicity. Keep your responses short and clear. Never use clinical or formal language. Always start your response with an avatar action tag ([smile], [nod], [concerned], [gentle], or [laugh]) followed by a short first utterance, then your reasoning in <think> tags, then your final response."}},
    {{"role": "user", "content": "..."}},
    {{"role": "assistant", "content": "[ACTION_TAG] [first utterance]\\n<think>\\n[reasoning]\\n</think>\\n[final response]"}}
  ],
  "metadata": {{
    "condition": "{condition}",
    "severity": "{severity}",
    "emotion": "{emotion}",
    "scenario": "{scenario}"
  }}
}}"""

FILTER_PROMPT = """Rate the following elderly care AI companion conversation from 1-10.

## Conversation
{conversation}

## Scoring Criteria
- Warmth & empathy: Does the AI respond with genuine care for the elderly user?
- Clarity: Are responses short, simple, and easy to understand?
- Domain fit: Does this feel like a natural dementia/Alzheimer's companion interaction?
- Voice quality: Natural spoken language, not clinical or robotic?

Score strictly. Reserve 8-10 for excellent conversations. Most should score 4-7.

## Output (JSON only, no markdown fences)
{{
  "score": <int 1-10>,
  "keep": <true if score >= 6>,
  "reason": "<one sentence>"
}}"""



REWRITE_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "rewritten_conversation",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "messages": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "role":    {"type": "string"},
                            "content": {"type": "string"},
                        },
                        "required": ["role", "content"],
                        "additionalProperties": False,
                    },
                },
                "metadata": {
                    "type": "object",
                    "properties": {
                        "condition": {"type": "string"},
                        "severity":  {"type": "string"},
                        "emotion":   {"type": "string"},
                        "scenario":  {"type": "string"},
                    },
                    "required": ["condition", "severity", "emotion", "scenario"],
                    "additionalProperties": False,
                },
            },
            "required": ["messages", "metadata"],
            "additionalProperties": False,
        },
    },
}

def sample_eq_params():
    return {k: random.choice(v) for k, v in EQ_MATRIX.items()}


def strip_think_blocks(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def format_conversation_for_filter(messages):
    lines = []
    for m in messages:
        if m["role"] == "system":
            continue
        content = strip_think_blocks(m["content"]) if m["role"] == "assistant" else m["content"]
        lines.append(f"{m['role'].upper()}: {content}")
    return "\n\n".join(lines)


def format_conversation_for_prompt(messages):
    return "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in messages if m["role"] != "system"
    )


def call_crof(system_prompt, user_prompt, temperature=0.8, max_tokens=2048,
              reasoning_effort="medium", response_format=None, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
                reasoning_effort=reasoning_effort,
                **({'response_format': response_format} if response_format is not None else {}),
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt+1}/{retries} in {wait}s: {e}")
                time.sleep(wait)
            else:
                print(f"  Failed after {retries} attempts: {e}")
                return None


def parse_json_response(raw):
    if raw is None:
        return None
    # Strip markdown fences
    cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    # Try direct parse first
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    # Fallback: extract first {...} block (handles preamble text)
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError as e:
            print(f"  JSON parse error: {e} | raw[:200]: {cleaned[:200]}")
    return None


print("Utilities ready.")

In [ ]:
# Cell 5: Layer 1 — EQ-Matrix rewrite pipeline (parallel)

REWRITE_SYSTEM = (
    "You are an expert dialogue rewriter specializing in dementia care AI companions. "
    "Rewrite conversations into natural dialogues between an elderly person with dementia "
    "and a caring AI companion. "
    "Always output valid JSON only. No preamble, no explanation, no markdown fences."
)


def rewrite_sample(sample, eq_params):
    original_conv = format_conversation_for_prompt(sample["messages"])
    prompt = REWRITE_PROMPT.format(
        condition=eq_params["condition"],
        severity=eq_params["severity"],
        emotion=eq_params["emotion"],
        scenario=eq_params["scenario"],
        character_name=CHARACTER["name"],
        character_personality=CHARACTER["personality"],
        original_conversation=original_conv,
    )
    raw = call_crof(REWRITE_SYSTEM, prompt, temperature=0.8, max_tokens=2048,
                   reasoning_effort="medium", response_format=REWRITE_RESPONSE_FORMAT)
    return parse_json_response(raw)


def rewrite_worker(args):
    idx, sample = args
    eq_params = sample_eq_params()
    result = rewrite_sample(sample, eq_params)
    return idx, result


rewritten_samples = [None] * len(samples)
failed_count = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(rewrite_worker, (i, s)): i for i, s in enumerate(samples)}
    for future in tqdm(as_completed(futures), total=len(samples), desc="Layer 1 rewrite"):
        idx, result = future.result()
        if result:
            rewritten_samples[idx] = result
        else:
            failed_count += 1

rewritten_samples = [s for s in rewritten_samples if s is not None]
print(f"Rewritten: {len(rewritten_samples):,} / {len(samples):,}  Failed: {failed_count}")

# Push Layer 1 output to HF Hub under rewritten/
login(token=HF_TOKEN)
Dataset.from_list(rewritten_samples).push_to_hub(
    HF_REPO_ID,
    data_dir="rewritten",
    private=True,
)
print(f"Pushed {len(rewritten_samples):,} rewritten samples to {HF_REPO_ID}/rewritten")

In [ ]:
# Cell 6: Layer 2 — Quality filter (parallel)

FILTER_SYSTEM = (
    "You are a quality evaluator for dementia care AI companion dialogues. "
    "Score conversations strictly and return ONLY valid JSON. No preamble, no explanation."
)


def filter_sample(sample):
    conv_str = format_conversation_for_filter(sample["messages"])
    prompt = FILTER_PROMPT.format(conversation=conv_str)
    raw = call_crof(FILTER_SYSTEM, prompt, temperature=0.1, max_tokens=200, reasoning_effort="medium")
    return parse_json_response(raw)


def filter_worker(sample):
    return sample, filter_sample(sample)


scored_samples = []
failed_count = 0
score_records = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(filter_worker, s): s for s in rewritten_samples}
    for future in tqdm(as_completed(futures), total=len(rewritten_samples), desc="Layer 2 score"):
        sample, scores = future.result()
        if scores is None:
            failed_count += 1
            continue
        score_records.append(scores)
        sample["quality_scores"] = scores
        scored_samples.append(sample)

avg_score = sum(s["score"] for s in score_records) / len(score_records) if score_records else 0
print(f"Scored: {len(scored_samples):,}  Failed: {failed_count}  Avg score: {avg_score:.1f}/10")

# Push all scored samples to HF Hub under scored/
login(token=HF_TOKEN)
Dataset.from_list(scored_samples).push_to_hub(
    HF_REPO_ID,
    data_dir="scored",
    private=True,
)
print(f"Pushed {len(scored_samples):,} scored samples to {HF_REPO_ID}/scored")

# --- Threshold selection: keep top N by score ---
TARGET = 9000
scored_samples.sort(key=lambda s: s["quality_scores"]["score"], reverse=True)
score_threshold = scored_samples[TARGET - 1]["quality_scores"]["score"] if len(scored_samples) >= TARGET else 1
filtered_samples = scored_samples[:TARGET]
print(f"\nTop {TARGET:,} samples (score >= {score_threshold}/10):")
dist = {}
for s in filtered_samples:
    v = s["quality_scores"]["score"]
    dist[v] = dist.get(v, 0) + 1
for k in sorted(dist, reverse=True):
    print(f"  score {k}: {dist[k]:,}")

Dataset.from_list(filtered_samples).push_to_hub(
    HF_REPO_ID,
    data_dir="filtered",
    private=True,
)
print(f"Pushed {len(filtered_samples):,} filtered samples to {HF_REPO_ID}/filtered")

In [ ]:
# Cell 7: Stats and sample inspection

conditions = [s["metadata"]["condition"] for s in filtered_samples]
severities = [s["metadata"]["severity"]  for s in filtered_samples]
emotions   = [s["metadata"]["emotion"]   for s in filtered_samples]
scenarios  = [s["metadata"]["scenario"]  for s in filtered_samples]

print("=== EQ-Matrix Distribution ===")
for label, data in [("Condition", conditions), ("Severity", severities),
                    ("Emotion", emotions), ("Scenario", scenarios)]:
    print(f"{label}: {dict(collections.Counter(data))}")

print("\n=== Random Sample ===")
sample = random.choice(filtered_samples)
for m in sample["messages"]:
    print(f"[{m['role'].upper()}]\n{m['content']}\n")
print("Metadata:", sample["metadata"])
print("Scores:",   sample.get("quality_scores", {}))

In [ ]:
# Cell 8: Final push — filtered dataset to HF Hub under filtered/
# This overwrites the Cell 6 push with quality_scores included.
# Note: Enable Gated access manually at
# https://huggingface.co/datasets/YUGOROU/amd-voice-sft-data/settings

hf_data = {
    "messages":       [s["messages"]               for s in filtered_samples],
    "metadata":       [s["metadata"]               for s in filtered_samples],
    "quality_scores": [s.get("quality_scores", {}) for s in filtered_samples],
}

Dataset.from_dict(hf_data).push_to_hub(
    HF_REPO_ID,
    data_dir="filtered",
    private=True,
)
print(f"Pushed {len(filtered_samples):,} samples to {HF_REPO_ID}/filtered")